In [2]:


def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    denominator = np.where(denominator == 0, 1e-8, denominator)
    return 100 * np.mean(np.abs(y_pred - y_true) / denominator)


def format_row_with_highlight(name, values, unit=""):
    """Highlight min (bold) and second min (underline) LaTeX-style."""
    values = np.array(values)
    idx_sorted = np.argsort(values)
    formatted = []

    for i, v in enumerate(values):
        val_str = f"{v:.2f}\\%" if unit == "%" else f"{v:.4f}"
        if i == idx_sorted[0]:
            val_str = r"\textbf{" + val_str + r"}"
        elif i == idx_sorted[1]:
            val_str = r"\underline{" + val_str + r"}"
        formatted.append(val_str)

    return f"        ${name}$ & " + " & ".join(formatted) + r" \\"


def generate_curv_table(base_path, file_paths, caption, save_path, labels=None):
    if labels is None:
        labels = [os.path.splitext(os.path.basename(p))[0] for p in file_paths]

    mse_list = []
    smape_list = []

    for path in file_paths:
        data = torch.load(os.path.join(base_path, path), weights_only=False)
        true_curv = data["curvatures_sub"][5].numpy()
        learned_curv = data["curvatures_sub"][6].numpy()

        mse = np.mean((learned_curv - true_curv) ** 2)
        smape_val = smape(true_curv, learned_curv)

        mse_list.append(mse)
        smape_list.append(smape_val)

    lines = []
    lines.append(r"\begin{table}[ht]")
    lines.append(r"    \centering")
    lines.append(f"    \\begin{{tabular}}{{{'l' * (len(file_paths) + 1)}}}")
    lines.append(r"        \toprule")
    lines.append("        & " + " & ".join(labels) + r" \\")
    lines.append(r"        \midrule")
    lines.append(format_row_with_highlight(r"\operatorname{MSE}", mse_list))
    lines.append(format_row_with_highlight(r"\operatorname{SMAPE}", smape_list, unit="%"))
    lines.append(r"        \bottomrule")
    lines.append(r"    \end{tabular}")
    lines.append(f"    \caption{{{caption}}}")
    lines.append(r"    \label{tab:curvature_metrics}")
    lines.append(r"\end{table}")

    with open(save_path, "w") as f:
        f.write("\n".join(lines))

    print(f"LaTeX table written to: {save_path}")


In [3]:
generate_curv_table(
    base_path="notebooks_m_vae/curvatures",
    file_paths=[
        "curvatures_exp00_flower_topo_VAE_100epochs.pt",
        "curvatures_exp01_flower_topo_VAE_100epochs.pt",
        "curvatures_exp02_flower_topo_VAE_100epochs.pt",        
        "curvatures_exp03_flower_topo_VAE_100epochs.pt",
        "curvatures_exp04_flower_topo_VAE_100epochs.pt",
        "curvatures_exp05_flower_topo_VAE_100epochs.pt",        
        "curvatures_exp06_flower_topo_VAE_100epochs.pt",
        "curvatures_exp07_flower_topo_VAE_100epochs.pt",
        "curvatures_exp08_flower_topo_VAE_100epochs.pt",        
        "curvatures_exp09_flower_topo_VAE_100epochs.pt",
        "curvatures_exp10_flower_topo_VAE_100epochs.pt",
        "curvatures_exp11_flower_topo_VAE_100epochs.pt",        
        "curvatures_exp12_flower_topo_VAE_100epochs.pt",
        "curvatures_exp13_flower_topo_VAE_100epochs.pt",
        "curvatures_exp14_flower_topo_VAE_100epochs.pt",
    ],
    caption="Curvature reconstruction metrics for different VAE configurations.",
    save_path="tables/curv_metrics_table.tex",
    labels=[
        r"$\beta=0,\gamma=-,d_{topo}=-$", r"$\beta=0.08,\gamma=-,d_{topo}=-$", r"$\beta=100,\gamma=-,d_{topo}=-$", 
        r"$\beta=0,\gamma=1,d_{topo}=0", r"$\beta=0.08,\gamma=1,d_{topo}=0$", r"$\beta=100,\gamma=1,d_{topo}=0$", 
        r"$\beta=0,\gamma=100,d_{topo}=0", r"$\beta=0.08,\gamma=100,d_{topo}=0$", r"$\beta=100,\gamma=100,d_{topo}=0$",  
        r"$\beta=0,\gamma=1,d_{topo}=1", r"$\beta=0.08,\gamma=1,d_{topo}=1$", r"$\beta=100,\gamma=1,d_{topo}=1$", 
        r"$\beta=0,\gamma=100,d_{topo}=1", r"$\beta=0.08,\gamma=100,d_{topo}=1$", r"$\beta=100,\gamma=100,d_{topo}=1$", 
        ]
)

NameError: name 'torch' is not defined

# VAE Complete Table

In [21]:
import json
import numpy as np


def format_value(val, unit="", highlight=None):
    if val == "-":
        return "-"
    v = f"{val:.2f}\\%" if unit == "%" else f"{val:.4f}"
    if highlight == "bold":
        return r"\textbf{" + v + "}"
    elif highlight == "underline":
        return r"\underline{" + v + "}"
    return v


def generate_curv_table_from_json_lookup(file_lookup, datasets, save_path):
    betas = [0, 0.08, 1]
    gammas = [0, 1, 100]
    dtopos = [0, 1, 2]

    # Step 1: collect full grid of data
    all_rows = []  # stores row-wise [meta, mse_values, smape_values]

    for beta in betas:
        for gamma in gammas:
            if gamma == 0 and beta != 0:
                continue
            for d in dtopos:
                if gamma == 0 and beta == 0 and d != 0:
                    continue

                meta = (f"{beta}", f"{gamma}", f"{d}")
                mse_values = []
                smape_values = []

                for ds in datasets:
                    path = file_lookup.get(ds, {}).get((beta, gamma, d), None)
                    if path is None:
                        mse_values.append("-")
                        smape_values.append("-")
                        continue
                    try:
                        with open(path, "r") as f:
                            json_data = json.load(f)
                        mse = json_data["errors"]["MSE"][6]
                        smape = json_data["errors"]["SMAPE_percent"][6]
                        mse_values.append(mse)
                        smape_values.append(smape)
                    except Exception as e:
                        print(f"Error reading {path}: {e}")
                        mse_values.append("-")
                        smape_values.append("-")

                all_rows.append((meta, mse_values, smape_values))

    # Step 2: compute column-wise highlights
    n_datasets = len(datasets)
    mse_column = [[] for _ in range(n_datasets)]
    smape_column = [[] for _ in range(n_datasets)]

    for row_id, (_, mse_vals, smape_vals) in enumerate(all_rows):
        for i in range(n_datasets):
            if mse_vals[i] != "-":
                mse_column[i].append((row_id, mse_vals[i]))
            if smape_vals[i] != "-":
                smape_column[i].append((row_id, smape_vals[i]))

    mse_highlight = dict()
    smape_highlight = dict()

    for i in range(n_datasets):
        if len(mse_column[i]) > 0:
            sorted_mse = sorted(mse_column[i], key=lambda x: x[1])
            mse_highlight[(sorted_mse[0][0], i)] = "bold"
            if len(sorted_mse) > 1:
                mse_highlight[(sorted_mse[1][0], i)] = "underline"
        if len(smape_column[i]) > 0:
            sorted_smape = sorted(smape_column[i], key=lambda x: x[1])
            smape_highlight[(sorted_smape[0][0], i)] = "bold"
            if len(sorted_smape) > 1:
                smape_highlight[(sorted_smape[1][0], i)] = "underline"

    # Step 3: render table
    lines = []
    lines.append(r"\begin{table}[ht]")
    lines.append(r"\scriptsize")
    lines.append(r"    \centering")
    lines.append(f"    \\begin{{tabular}}{{lll|{'cc|' * n_datasets}}}")
    lines.append(r"        \toprule")
    header = "        $\\beta$ & $\\gamma$ & $d_{\\text{topo}}$"
    for name in datasets:
        header += f" & \\multicolumn{{2}}{{c|}}{{{name}}}"
    lines.append(header + r" \\")
    lines.append("        & & & " + " & ".join(["MSE & SMAPE"] * n_datasets) + r" \\")
    lines.append(r"        \midrule")

    for row_id, (meta, mse_vals, smape_vals) in enumerate(all_rows):
        alpha, gamma, dtopo = meta
        dtopo_display = "-" if gamma == "0" else dtopo
        row = [alpha, gamma, dtopo_display]
        for i in range(n_datasets):
            mse_fmt = format_value(mse_vals[i], highlight=mse_highlight.get((row_id, i)))
            smape_fmt = format_value(smape_vals[i], unit="%", highlight=smape_highlight.get((row_id, i)))
            row.append(mse_fmt)
            row.append(smape_fmt)
        lines.append("        " + " & ".join(row) + r" \\")

    lines.append(r"        \bottomrule")
    lines.append(r"    \end{tabular}")
    lines.append(r"    \caption{Curvature metrics across models and datasets.}")
    lines.append(r"    \label{tab:curv_full}")
    lines.append(r"\end{table}")

    with open(save_path, "w") as f:
        f.write("\n".join(lines))

    print(f"LaTeX table written to: {save_path}")


In [22]:
base_path_flower = "notebooks_m_vae/results/s1_synthetic/"
base_path_scrunchy = "notebooks_m_vae/results/scrunchy_dim_n/"
file_lookup = {
    "Curve low dim": {
        (0, 0, 0): base_path_flower + "results_exp00_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0.08, 0, 0): base_path_flower + "results_exp01_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (1, 0, 0): base_path_flower + "results_exp02_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0, 1, 0): base_path_flower + "results_exp03_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0.08, 1, 0): base_path_flower + "results_exp04_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (1, 1, 0): base_path_flower + "results_exp05_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0, 100, 0): base_path_flower + "results_exp06_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0.08, 100, 0): base_path_flower + "results_exp07_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (1, 100, 0): base_path_flower + "results_exp08_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0, 1, 1): base_path_flower + "results_exp09_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0.08, 1, 1): base_path_flower + "results_exp10_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (1, 1, 1): base_path_flower + "results_exp11_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0, 100, 1): base_path_flower + "results_exp12_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0.08, 100, 1): base_path_flower + "results_exp13_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (1, 100, 1): base_path_flower + "results_exp14_flower_topo_VAE_100epochs/curvature_errors_stats.json",
    },
    "Curve high dim": {
        (0, 0, 0): base_path_scrunchy + "results_exp00_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0.08, 0, 0): base_path_scrunchy + "results_exp01_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (1, 0, 0): base_path_scrunchy + "results_exp02_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0, 1, 0): base_path_scrunchy + "results_exp03_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0.08, 1, 0): base_path_scrunchy + "results_exp04_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (1, 1, 0): base_path_scrunchy + "results_exp05_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0, 100, 0): base_path_scrunchy + "results_exp06_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0.08, 100, 0): base_path_scrunchy + "results_exp07_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (1, 100, 0): base_path_scrunchy + "results_exp08_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0, 1, 1): base_path_scrunchy + "results_exp09_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0.08, 1, 1): base_path_scrunchy + "results_exp10_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (1, 1, 1): base_path_scrunchy + "results_exp11_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0, 100, 1): base_path_scrunchy + "results_exp12_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0.08, 100, 1): base_path_scrunchy + "results_exp13_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (1, 100, 1): base_path_scrunchy + "results_exp14_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
    },
}

datasets = [
    "Curve low dim", "Curve high dim",
    "Sphere low dim", "Sphere high dim",
    "Torus low dim", "Torus high dim"
]

generate_curv_table_from_json_lookup(
    file_lookup=file_lookup,
    datasets=datasets,
    save_path="tables/m_vae_full_table.tex"
)

LaTeX table written to: tables/m_vae_full_table.tex
